# Closed loop: LUT vs NN
$[\theta_{ref}]$ → $[e=\theta_{ref}-\theta(t-\Delta)]$ → [PID] → $[\tau_{fb}]$

$\tau_{req}=\tau_{fb}+\tau_{ff}$ → LUT or NN → $\vec u$ → MyoSuite/MuJoCo → $\theta$.

$\Delta$ = `DELAY_MS` (SNN of near-nes: 150 ms); $\tau_{ff}$ from the inverse dynamics of the minimum-jerk trajectory (none for the step).

## Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

import numpy as np

from src.sim.trajectory import Trajectory
from src.lut.lut import Lut
from src.lut.lut_generator import LutGenerator
from src.nn.nn import NeuralNetwork

from src.sim.elbow_simulation import load_model, simulate_closed_loop, feedforward_torque, pid_grid_search
from src.sim.simulation_io import (simulation_path, simulation_title, save_simulations,
                                   load_simulations, load_reference,
                                   plot_angle, plot_torque, plot_activations)

## Trajectory Generation

In [ ]:
TRAJECTORY = "minjerk" # "step"

RESOLUTION_MS = 1.0 # [ms] also the MuJoCo timestep
TIME_PREP_MS = 100.0 # [ms]

init_angle = 0 # [°]
angle_ref = 130 # [°]

# for "step"
STEP_DURATION_MS = 500.0 # [ms]

# for "minjerk"
TIME_MOVE_MS = 500.0 # [ms]
TIME_LOCKED_MS = 150.0 # [ms]
TIME_POST_MS = 3000.0 # [ms]

if TRAJECTORY == "step":
    traj = Trajectory.step(init_angle, angle_ref, duration_ms=STEP_DURATION_MS,
                           resolution_ms=RESOLUTION_MS, time_prep_ms=TIME_PREP_MS)
elif TRAJECTORY == "minjerk":
    traj = Trajectory.minjerk(init_angle, angle_ref, resolution_ms=RESOLUTION_MS,
                              time_prep_ms=TIME_PREP_MS, time_move_ms=TIME_MOVE_MS,
                              time_locked_ms=TIME_LOCKED_MS, time_post_ms=TIME_POST_MS,
                              hold_in_post=True)
else:
    raise ValueError(f"TRAJECTORY: {TRAJECTORY} not valid.")
print(traj)
traj.plot(show_phases=True)

## Parameters

In [ ]:
W_ref = 0.5
P_VALUE = 3
MODELS_STRUCTURE = ["2L_64U_300E"]  # "2L_32U", "2L_64U", "3L_64U"
ACTIVATION_MODE = ["relu"]
OUTPUT_ACTIVATION_MODE = ["linear_cosdecay"]

PATH_LUT = f"models/luts/W_all/p{P_VALUE}/lut_p{P_VALUE}_W_all.nc"
PATH_MUJOCO_MODEL = "assets/myo_sim/elbow/myoelbow_1dof6muscles.xml"

PATH_MODELS = [f"models/nn/W_all/grid_search_p{P_VALUE}/{struct}_{act}_{out}"
               for struct in MODELS_STRUCTURE
               for act in ACTIVATION_MODE
               for out in OUTPUT_ACTIVATION_MODE]
print(PATH_MODELS)

TIMESTEP = traj.dt_s  # [s] MuJoCo timestep = resolution of the trajectory

## Upload LUT, NN and MuJoCo model

In [ ]:
my_lut = Lut.load(PATH_LUT)

mj_model = load_model(PATH_MUJOCO_MODEL, timestep=TIMESTEP)

my_lut_generator = LutGenerator(PATH_MUJOCO_MODEL, p_value=P_VALUE, w_values=[W_ref])

allocators = {
    "LUT":    my_lut.get_activations,
    "solver": lambda W, theta, theta_dot, tau: my_lut_generator.get_activations(W, theta, tau, theta_dot),
}
allocator_paths = {"LUT": PATH_LUT,
                   "solver": f"{PATH_MUJOCO_MODEL} (online, p={P_VALUE:g})"}

my_nn_models = []
for path in PATH_MODELS:
    nn = NeuralNetwork.load(path)
    my_nn_models.append(nn)
    allocators[nn.nn_name] = nn.get_activations
    allocator_paths[nn.nn_name] = path

## Simulation

In [ ]:
K_P = 1.0  # 0.039
K_I = 1.0  # 0.006
K_D = 0.0  # 0.009

DELAY_MS = 150.0     # [ms] delay of the feedback in the PID (SNN: 150 ms)
FEEDFORWARD = True   # tau_ff from the inverse dynamics of theta_d (only minjerk)
SAVE = True

use_ff = FEEDFORWARD and traj.kind == "minjerk"
save_path = simulation_path(traj, W_ref, K_P, K_I, K_D, delay_ms=DELAY_MS, ff=use_ff)
title = simulation_title(traj, W_ref, K_P, K_I, K_D, delay_ms=DELAY_MS, ff=use_ff)

sims = {}
for name, allocator in allocators.items():
    print(f"Simulating: {name}")
    sims[name] = simulate_closed_loop(mj_model, allocator, W_ref, traj, K_P=K_P, K_I=K_I, K_D=K_D,
                                      delay_ms=DELAY_MS, feedforward=use_ff)

if SAVE:
    parameters = dict(W=W_ref, K_P=K_P, K_I=K_I, K_D=K_D, delay_ms=DELAY_MS, ff=use_ff,
                      trajectory=traj.params, timestep=TIMESTEP, model=PATH_MUJOCO_MODEL,
                      allocators=allocator_paths)
    save_simulations(sims, save_path, reference=traj.data, parameters=parameters)

## Plots

In [ ]:
SAVE_PLOTS = False
save_dir = save_path if SAVE_PLOTS else None

plot_angle(sims, title=f"Elbow angle ({title})", show_phases=True, save_dir=save_dir)
plot_torque(sims, title=f"Torque ({title})", save_dir=save_dir)
plot_activations(sims, title=f"Activations ({title})", save_dir=save_dir)
plot_activations(sims, title=f"Activation states ({title})", state=True, save_dir=save_dir)

# zoom on the movement:
# plot_torque(sims, title=f"Torque ({title})", t_min=0.0, t_max=1.0, save_dir=save_dir)

## PID Tuning

In [ ]:
PID_ALLOCATOR = "solver"   # "LUT" or the name of an NN

results = pid_grid_search(mj_model, allocators[PID_ALLOCATOR], W_ref, traj,
                          K_P_values=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
                          K_I_values=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
                          K_D_values=[0.0, 0.1, 0.2],
                          delay_ms=DELAY_MS, feedforward=use_ff)
results.head(10)